# Практика · Багатомовність і українська

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє: [homework.html](homework.html)

> ⏱ Зошит навчає **два десятки субслівних словників** і **три десятки лінійних
> моделей**. Заміряно: **близько 140 секунд процесорного часу** на чотирьох ядрах без
> відеокарти, **в один потік**. Стінного часу буде більше, і залежить воно від
> того, чим ще зайнята машина: на завантаженій машині наші прогони брали
> від пʼяти до семи хвилин. Останньою клітинкою зошит друкує власний процесорний час — звір
> із цим числом.

## Задача зошита

У нас є **паралельний** корпус: кожен запис — англійський оригінал і його
український переклад, тобто **той самий зміст двома мовами**. Це дає змогу
поставити питання, яке інакше поставити ніде:

1. **Скільки коштує українська в токенах** — не «взагалі», а на тому самому
   реченні, що й англійська.
2. **Що насправді спільного** в двох мов усередині спільного субслівного
   словника — і скільки з цієї спільності уявне.
3. **Чи працює крос-мовний перенос**: навчимо класифікатор на англійському
   боці й перевіримо його на українському, жодного разу не показавши йому
   українських міток.

Мітка «це повідомлення про помилку» береться регуляркою по **англійському**
оригіналу. Тому вона однаково справедлива для обох боків пари, а модель
українського боку її англійського джерела не бачить.

⚠️ **Готових багатомовних ваг у нас немає** — мережа в зошитах курсу
заборонена, кеш порожній. Усе, що тут є, ми навчаємо самі з нуля. Це не
mBERT і не XLM-R; це найдешевша модель того самого механізму, і саме тому
в ній видно, звідки береться перенос.

## 0 · Середовище й чому ми фіксуємо потоки

Зошит міряє час, а час на спільній машині бреше двома способами.

**Стінний годинник** показує, скільки минуло реального часу. Якщо поряд
рахується щось іще — він покаже більше, і число нічого не варте.

**Процесорний годинник** (`time.process_time()`) рахує тільки той час, коли
процесор працював над нашою програмою. Але й у нього є пастка: якщо
бібліотека лінійної алгебри розкладає роботу на потоки, то потоки, які
**чекають**, теж зараховуються як робота. Тому спершу фіксуємо один потік —
і робимо це **до** імпорту `numpy`, бо змінні середовища читаються під час
завантаження бібліотеки.

In [ ]:
import os
# ⚠️ ці три рядки мусять стояти ДО імпорту numpy — інакше бібліотеки
# вже запустять свої потоки, і процесорний час стане брехливим
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import glob, gettext, re, math, time, sys, gc
from collections import Counter
import numpy as np

ЗАПУСК = time.process_time()          # звідси рахуємо власний час зошита

print('python              ', sys.version.split()[0])
print('numpy               ', np.__version__)
print('ядер у машині       ', os.cpu_count())
print('навантаження машини ', round(os.getloadavg()[0], 2))

## 1 · Паралельний корпус

Беремо українські переклади інтерфейсів із `/usr/share/locale/uk/LC_MESSAGES/`.
Кожен запис каталогу — це пара «англійський оригінал → український переклад».
Досі курс брав звідти лише український бік; сьогодні нам потрібні **обидва**,
і саме тому, що вони описують **той самий зміст**.

Слова рахуємо двома регулярними виразами — по одному на мову. Український —
канонічний для курсу: апостроф у ньому не символ слова, а **звʼязка**
всередині слова, тож `зʼєднання` лишається одним словом.

Відсіюємо надто короткі й надто довгі рядки: у парі з одного слова нема чого
міряти, а хвіст на півтисячі слів перекосив би середні.

In [ ]:
UK_WORD = re.compile(r"[а-яїієґ]+(?:[\'ʼ’][а-яїієґ]+)*")
EN_WORD = re.compile(r"[a-z]+(?:\'[a-z]+)*")

# мітка беремо з АНГЛІЙСЬКОГО оригіналу — вісім типових слів помилки
ERROR_WORDS = re.compile(
    r'\b(error|failed|cannot|could not|unable|invalid|denied|no such)\b', re.I)


def load_pairs():
    """Повертає список (програма, англійський оригінал, український переклад)."""
    out = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as f:
                catalog = gettext.GNUTranslations(f)
        except Exception:
            continue                     # зламаний каталог просто пропускаємо
        program = path.split('/')[-1][:-3]
        for source, target in catalog._catalog.items():
            # службовий заголовок каталогу має ключ '' і текстом не є
            if isinstance(source, str) and isinstance(target, str) \
               and len(target) > 30 and 'Project-Id' not in target:
                en, uk = source.lower(), target.lower()
                n_en, n_uk = len(EN_WORD.findall(en)), len(UK_WORD.findall(uk))
                if 2 <= n_en <= 40 and 2 <= n_uk <= 40:
                    out.append((program, en, uk, n_en, n_uk))
    return out


rows = load_pairs()
if len(rows) < 1000:
    raise RuntimeError('українських каталогів у системі майже немає — '
                       'зошитові немає на чому працювати')

en_words = sum(r[3] for r in rows)
uk_words = sum(r[4] for r in rows)
en_chars = sum(len(r[1]) for r in rows)
uk_chars = sum(len(r[2]) for r in rows)

print('пар «оригінал — переклад»', len(rows))
print('програм                  ', len({r[0] for r in rows}))
print()
print('%-22s %10s %10s %8s' % ('', 'англійська', 'українська', 'укр/англ'))
print('%-22s %10d %10d %8.4f' % ('слововживань', en_words, uk_words,
                                 uk_words / en_words))
print('%-22s %10d %10d %8.4f' % ('символів', en_chars, uk_chars,
                                 uk_chars / en_chars))
print('%-22s %10d %10d %8.4f' % (
    'різних словоформ',
    len({w for r in rows for w in EN_WORD.findall(r[1])}),
    len({w for r in rows for w in UK_WORD.findall(r[2])}),
    len({w for r in rows for w in UK_WORD.findall(r[2])}) /
    len({w for r in rows for w in EN_WORD.findall(r[1])})))

Перший висновок уже тут, і він тримає всю тему: **на тому самому змісті
українська вживає менше слів, але має їх більше різних**. Менше слів — бо
українська обходиться без артиклів і частини службових слів; більше різних —
бо кожне слово існує в багатьох формах.

## 2 · Поділ: перевірна, відкладена, навчальна

Ділимо на **три** частини:

- **перевірна** — на ній оголошується результат, модель її не бачить ніколи;
- **відкладена** — на ній добирають гіперпараметри (ми свої дібрали окремим
  прогоном, про це нижче, але поділ лишаємо такий самий, як у доборі);
- **навчальна** — на ній усе вчиться.

⚠️ Рядки лежать **у порядку програм**: спершу всі рядки `abrt`, потім
`accountsservice` і так далі. Якби ми взяли «перші десять відсотків», це була б
не менша вибірка, а **вужчий домен** — пів десятка програм замість двохсот
шістдесяти. Тому перемішуємо з фіксованим зерном.

Навчальну частину ще й обрізаємо до 30 000 пар — це той бюджет, на якому
добирались гіперпараметри, і той, на якому зошит укладається в три хвилини.

In [ ]:
V_SHARED = 8000        # розмір субслівного словника
N_TRAIN = 30000        # стільки пар бачить кожна модель


def split_indices(seed):
    """Три непересічні набори позицій. Перемішування — обовʼязкове."""
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(rows))
    n = len(rows) // 10
    return order[:n], order[n:2 * n], order[2 * n:2 * n + N_TRAIN]


test_i, dev_i, train_i = split_indices(0)
labels = np.array([1 if ERROR_WORDS.search(r[1]) else 0 for r in rows])

print('перевірних ', len(test_i))
print('відкладених', len(dev_i))
print('навчальних ', len(train_i))
print()
print('частка класу «помилка» у всьому корпусі', round(float(labels.mean()), 4))
print('частка класу «помилка» у перевірній    ',
      round(float(labels[test_i].mean()), 4))

## 3 · Спільний субслівний словник

Багатомовна модель має **один** словник шматків на всі мови. Навчимо такий
самі: візьмемо BPE (той самий алгоритм, що в темі
[02](../02-tokenization/lecture.html)) і покажемо йому обидва боки корпусу
одразу.

Для порівняння навчимо ще два **одномовні** словники того самого розміру —
англійський і український. Далі кожен вимір робитимемо в усіх трьох.

⚠️ Пре-токенізатор беремо `Whitespace`, а не наш словесний вираз: справжня
багатомовна модель бачить сирий текст із цифрами, відсотками й `%s`, і саме
ця обставина буде важлива в розділі 6.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers


def train_bpe(texts, vocab_size):
    """Навчає BPE на переданих рядках. Повертає готовий токенізатор."""
    tok = Tokenizer(models.BPE(unk_token='[UNK]'))
    tok.pre_tokenizer = pre_tokenizers.Whitespace()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, min_frequency=2,
                                  special_tokens=['[PAD]', '[UNK]'],
                                  show_progress=False)
    tok.train_from_iterator(texts, trainer)
    return tok


en_train_text = [rows[i][1] for i in train_i]
uk_train_text = [rows[i][2] for i in train_i]

started = time.process_time()
tok_en = train_bpe(en_train_text, V_SHARED)
tok_uk = train_bpe(uk_train_text, V_SHARED)
tok_shared = train_bpe(en_train_text + uk_train_text, V_SHARED)
print('три словники навчено за %.1f с процесорного часу'
      % (time.process_time() - started))

Перше питання до спільного словника: **кому дісталося місце**. Розберемо
всі 8000 шматків за тим, з яких літер вони складаються.

In [ ]:
def alphabet_of(piece):
    """До якої абетки належить шматок словника."""
    latin = any('a' <= c <= 'z' for c in piece)
    cyrillic = any('Ѐ' <= c <= 'ӿ' for c in piece)
    if latin and cyrillic:
        return 'обидві абетки'
    if latin:
        return 'латиниця'
    if cyrillic:
        return 'кирилиця'
    return 'без літер'


split_by_alphabet = Counter(alphabet_of(p) for p in tok_shared.get_vocab())
print('спільний словник на', tok_shared.get_vocab_size(), 'шматків:')
for name, count in split_by_alphabet.most_common():
    print('  %-14s %5d  %5.2f %%' % (name, count,
                                     count / tok_shared.get_vocab_size() * 100))

Половина словника — кирилиця, половина — латиниця. Це і є **механізм
прокляття багатомовності** в найпростішому вигляді: місце скінченне, і кожна
нова мова відкушує свою частку в решти.

Подивімось, як це виглядає на конкретному слові.

In [ ]:
PROBE = ['налаштування', 'зʼєднання', 'помилка', 'неможливо',
         'settings', 'connection', 'error', 'impossible']

print('%-16s %-34s %s' % ('слово', 'власний словник мови', 'спільний словник'))
for word in PROBE:
    own = tok_uk if UK_WORD.fullmatch(word) else tok_en
    a = ' '.join(own.encode(word).tokens)
    b = ' '.join(tok_shared.encode(word).tokens)
    print('%-16s %-34s %s' % (word, a, b))

## 4 · Націнка в токенах: скільки коштує те саме речення

Тепер головний вимір теми. Беремо **перевірну** частину — ті самі 9166 пар —
і рахуємо, скільки токенів виходить з англійського боку й скільки з
українського. Зміст той самий, тож усе, що ми побачимо, — це різниця мов і
словників, а не різниця текстів.

Робимо це для п'яти розмірів словника й для трьох способів його навчити.

In [ ]:
SIZES = [2000, 4000, 8000, 16000, 32000]

test_en = [rows[i][1] for i in test_i]
test_uk = [rows[i][2] for i in test_i]
w_en = sum(rows[i][3] for i in test_i)     # слів у перевірній, англійський бік
w_uk = sum(rows[i][4] for i in test_i)     # ...і український


def total_tokens(tok, texts):
    """Скільки всього токенів дає токенізатор на цих рядках."""
    return sum(len(e.ids) for e in tok.encode_batch(texts))


started = time.process_time()
tokens = {}                                # (розмір, чий словник) -> (англ, укр)
alphabet = {}                              # розмір -> як спільний словник поділився
for size in SIZES:
    for name, texts in [('англійський', en_train_text),
                        ('український', uk_train_text),
                        ('спільний', en_train_text + uk_train_text)]:
        tok = train_bpe(texts, size)
        tokens[(size, name)] = (total_tokens(tok, test_en),
                                total_tokens(tok, test_uk))
        if name == 'спільний':
            alphabet[size] = Counter(alphabet_of(p) for p in tok.get_vocab())
        del tok
gc.collect()
print('перебір словників зайняв %.1f с процесорного часу'
      % (time.process_time() - started))
print('слів у перевірній: англійських %d, українських %d' % (w_en, w_uk))

In [ ]:
print('ТОКЕНІВ НА 1000 СЛІВ СВОЄЇ МОВИ')
print('%8s | %9s %9s %9s | %9s %9s %9s' %
      ('словник', 'англ', 'укр', 'націнка', 'англ', 'укр', 'націнка'))
print('%8s | %29s | %29s' % ('', 'власний словник мови', 'спільний словник'))
for size in SIZES:
    e_own = tokens[(size, 'англійський')][0] / w_en * 1000
    u_own = tokens[(size, 'український')][1] / w_uk * 1000
    e_sh, u_sh = tokens[(size, 'спільний')]
    e_sh = e_sh / w_en * 1000
    u_sh = u_sh / w_uk * 1000
    print('%8d | %9.1f %9.1f %8.1f%% | %9.1f %9.1f %8.1f%%' %
          (size, e_own, u_own, (u_own / e_own - 1) * 100,
           e_sh, u_sh, (u_sh / e_sh - 1) * 100))

І одразу — як спільний словник ділиться між абетками, коли йому дають
різне місце. Це та сама величина, що в розділі 3, але для всіх пʼятьох
розмірів.

In [ ]:
print('ЯК ДІЛИТЬСЯ СПІЛЬНИЙ СЛОВНИК')
print('%8s %10s %10s %12s' % ('розмір', 'кирилиця', 'латиниця', 'без літер'))
for size in SIZES:
    a = alphabet[size]
    print('%8d %10d %10d %12d' % (size, a['кирилиця'], a['латиниця'],
                                  a['без літер'] + a['обидві абетки']))

Читай таблицю так: у стовпчику «націнка» стоїть, на скільки відсотків
більше токенів витрачає українська на **тисячу своїх слів**. Число велике —
і воно **падає, коли словнику дають місце**. Це головна практична порада теми:
скарга «модель погано працює з українською» дуже часто означає «українській
дали замало місця у словнику».

А тепер той самий зміст без перерахунку на слова: скільки токенів іде на
**один запис** кожною мовою. Тут порівняння найчистіше — це буквально одне
й те саме речення.

In [ ]:
print('ТОЙ САМИЙ ЗМІСТ: у скільки разів українська дорожча за англійську')
print('%8s %14s %14s' % ('словник', 'власні', 'спільний'))
for size in SIZES:
    own = tokens[(size, 'український')][1] / tokens[(size, 'англійський')][0]
    e_sh, u_sh = tokens[(size, 'спільний')]
    print('%8d %14.4f %14.4f' % (size, own, u_sh / e_sh))

## 5 · Що коштує сусід: прокляття багатомовності на словнику

Порівняємо два світи з **однаковим** бюджетом словника: у першому мова живе
сама, у другому ділить те саме місце із сусідкою. Різниця й буде ціною
багатомовності — заміряною, а не переказаною.

In [ ]:
print('НА СКІЛЬКИ БІЛЬШЕ ТОКЕНІВ ПІСЛЯ ПЕРЕЇЗДУ В СПІЛЬНИЙ СЛОВНИК')
print('%8s %12s %12s' % ('словник', 'англійська', 'українська'))
for size in SIZES:
    e_sh, u_sh = tokens[(size, 'спільний')]
    print('%8d %11.2f%% %11.2f%%' %
          (size, (e_sh / tokens[(size, 'англійський')][0] - 1) * 100,
           (u_sh / tokens[(size, 'український')][1] - 1) * 100))

## 6 · А якщо словника для твоєї мови немає зовсім

Крайній випадок тієї самої історії. Візьмемо український текст і поріжемо
його **англійським** словником — так поводиться модель, у чиєму
передтренуванні твоєї мови не було. І навпаки.

In [ ]:
print('%-42s %10s %8s' % ('що ріжемо і чим', 'токенів', 'разів'))
u_own = tokens[(8000, 'український')][1]
e_own = tokens[(8000, 'англійський')][0]
u_foreign = tokens[(8000, 'англійський')][1]
e_foreign = tokens[(8000, 'український')][0]
e_shared, u_shared = tokens[(8000, 'спільний')]

print('%-42s %10d %8.2f' % ('укр текст, український словник', u_own, 1.0))
print('%-42s %10d %8.2f' % ('укр текст, спільний словник', u_shared,
                            u_shared / u_own))
print('%-42s %10d %8.2f' % ('укр текст, АНГЛІЙСЬКИЙ словник', u_foreign,
                            u_foreign / u_own))
print()
print('%-42s %10d %8.2f' % ('англ текст, англійський словник', e_own, 1.0))
print('%-42s %10d %8.2f' % ('англ текст, спільний словник', e_shared,
                            e_shared / e_own))
print('%-42s %10d %8.2f' % ('англ текст, УКРАЇНСЬКИЙ словник', e_foreign,
                            e_foreign / e_own))

Числа несиметричні, і це не випадковість: у наших українських рядках
латиниця трапляється (`%s`, назви програм, скорочення), а кирилиця в
англійських — майже ніколи. Тому український словник частково знає англійську,
а англійський українську — ні.

## 7 · Що насправді спільного в двох мов у спільному словнику

Порахуймо, якими шматками користується кожна мова на перевірній частині — і
скільки шматків спільні.

In [ ]:
enc_en = tok_shared.encode_batch(test_en)
enc_uk = tok_shared.encode_batch(test_uk)
use_en, use_uk = Counter(), Counter()
for e in enc_en:
    use_en.update(e.ids)
for e in enc_uk:
    use_uk.update(e.ids)

both = set(use_en) & set(use_uk)
total_en, total_uk = sum(use_en.values()), sum(use_uk.values())

print('шматків ужито англійською      ', len(use_en))
print('шматків ужито українською      ', len(use_uk))
print('ужито обома мовами             ', len(both))
print()
print('частка ВЖИВАНЬ спільними шматками: англ %.4f  укр %.4f' %
      (sum(use_en[i] for i in both) / total_en,
       sum(use_uk[i] for i in both) / total_uk))

Третина українських вживань припадає на спільні шматки — звучить
непогано. Але подивімось, **що це за шматки**.

In [ ]:
id_to_piece = {i: p for p, i in tok_shared.get_vocab().items()}
ordered = sorted(both, key=lambda i: -(use_en[i] + use_uk[i]))

print('%-12s %8s %8s' % ('шматок', 'англ', 'укр'))
for i in ordered[:14]:
    print('%-12s %8d %8d' % (repr(id_to_piece[i]), use_en[i], use_uk[i]))

with_letters = [i for i in both if alphabet_of(id_to_piece[i]) != 'без літер']
print()
print('спільних шматків із хоч однією літерою: %d із %d'
      % (len(with_letters), len(both)))
print('частка вживань саме такими шматками: англ %.4f  укр %.4f' %
      (sum(use_en[i] for i in with_letters) / total_en,
       sum(use_uk[i] for i in with_letters) / total_uk))

Ось де спільність виявляється **уявною**. Верхівка списку — це відсоток,
крапка, кома й дефіс; а спільні шматки з літерами, які трапляються в
українському тексті, — це англійські слова `to`, `the`, `not`, `file`, тобто
неперекладені шматки інтерфейсу, а не спільна мовна тканина.

## 8 · Апостроф: одна літера, три способи її записати

Дрібниця, яка коштує української дорого. Апостроф у `зʼєднання` можна
записати щонайменше трьома різними символами Unicode, і токенізатор бачить
три різні слова.

In [ ]:
APOSTROPHES = {'ʼ (U+02BC, модифікатор)': '\u02bc',
               '’ (U+2019, права лапка)': '\u2019',
               "' (U+0027, машинописний)": "\u0027"}

uk_all = ' '.join(rows[i][2] for i in range(len(rows)))
print('скільки разів кожен символ трапився в українському боці корпусу:')
for name, ch in APOSTROPHES.items():
    print('  %-28s %7d' % (name, uk_all.count(ch)))

print()
print('як спільний словник ріже те саме слово в кожному написанні:')
for name, ch in APOSTROPHES.items():
    word = 'з' + ch + 'єднання'
    print('  %-28s %s' % (name, ' '.join(tok_shared.encode(word).tokens)))
del uk_all
_ = gc.collect()

## 9 · Чому українська роздуває словник: закон Гіпса на паралельному змісті

Закон Гіпса каже: кількість **різних** слів росте як обсяг тексту в степені
β, де β між нулем і одиницею. Що більше β, то швидше словник роздувається.

Заміряємо його для обох мов **на тому самому змісті** — це і є перевага
паралельного корпусу. Перемішуємо записи (три зерна), йдемо потоком слів і
дивимось, скільки різних уже трапилось.

In [ ]:
POINTS = [1000, 3000, 10000, 30000, 100000, 300000]


def heaps_beta(column, pattern, seed):
    """Показник Гіпса: нахил прямої log(словник) від log(слововживань)."""
    rng = np.random.default_rng(seed)
    stream = []
    for i in rng.permutation(len(rows)):
        stream.extend(pattern.findall(rows[i][column]))
        if len(stream) >= POINTS[-1]:
            break
    seen, xs, ys, sizes = set(), [], [], []
    for k, word in enumerate(stream[:POINTS[-1]], 1):
        seen.add(word)
        if k in POINTS:
            xs.append(math.log(k)); ys.append(math.log(len(seen)))
            sizes.append(len(seen))
    return float(np.polyfit(xs, ys, 1)[0]), sizes


started = time.process_time()
for lang, column, pattern in [('англійська', 1, EN_WORD),
                              ('українська', 2, UK_WORD)]:
    betas, sizes0 = [], None
    for seed in range(3):
        beta, sizes = heaps_beta(column, pattern, seed)
        betas.append(beta)
        if seed == 0:
            sizes0 = sizes
    print('%-12s beta = %.4f ±%.4f   словник у точках %s'
          % (lang, float(np.mean(betas)), float(np.std(betas)), sizes0))
print('слововживань у точках', POINTS)
print('зайняло %.1f с процесорного часу' % (time.process_time() - started))

### 9.1 · Наскільки взагалі важче вгадати українське слово

Останній вимір перед головним. **Уніграмний рубіж** — це середня несподіванка
токена для найдурнішої з можливих моделей: тієї, що не дивиться на контекст
узагалі, а знає лише частоти. Міряємо його в **натах** — це та сама
несподіванка, порахована через натуральний логарифм.

Число корисне двічі. По-перше, воно показує, наскільки українська менш
передбачувана на тому самому змісті. По-друге, це **поріг**, нижче якого мусить
опуститись будь-яка мовна модель, перш ніж її взагалі варто з чимось
порівнювати: доки не опустилась — моделі фактично ще немає.

In [ ]:
def unigram_bits(column):
    """Середня несподіванка токена в натах для моделі «лише частоти»."""
    counts = Counter()
    for e in tok_shared.encode_batch([rows[i][column] for i in train_i]):
        counts.update(e.ids)
    total, width = sum(counts.values()), tok_shared.get_vocab_size()
    logp, n = 0.0, 0
    for e in tok_shared.encode_batch([rows[i][column] for i in test_i]):
        for piece in e.ids:
            # додаємо одиницю до кожного лічильника, щоб ніколи не ділити на нуль
            logp += math.log((counts[piece] + 1) / (total + width))
            n += 1
    return -logp / n, n


for lang, column in [('англійська', 1), ('українська', 2)]:
    value, n = unigram_bits(column)
    print('%-12s уніграмний рубіж %.4f ната  (%d токенів перевірної)'
          % (lang, value, n))

## 10 · Крос-мовний перенос: головне питання теми

Обіцянка багатомовної моделі звучить так: **навчи задачу на мові, де мітки є,
і вона працюватиме там, де міток немає**. Перевіримо це прямо.

Мітка в нас береться з англійського оригіналу, тож вона є для обох боків
пари — але класифікаторові ми покажемо мітки **лише однієї** мови, а
питатимемо його про **іншу**.

Порівняємо три режими. Усім трьом дістається однакове: той самий спільний
словник, той самий обсяг даних, та сама лінійна модель, та сама сітка добору.

| режим | що спільного в мов |
|---|---|
| **A · лише словник** | шматки спільні, більше нічого |
| **B · простір із пар** | шматки спільні **плюс** розклад, навчений на парах «оригінал — переклад» |
| **B0 · простір нарізно** | шматки спільні плюс розклад того самого розміру, але навчений на тих самих текстах, розкладених по **різних** рядках |

B0 — це контроль. Він відрізняється від B **лише** тим, чи стояли переклад і
оригінал в одному рядку, коли будувався простір. Якщо перенос дає B і не дає
B0, значить справа не в стисканні, а саме в **звʼязку між мовами**.

### 10.1 · Як будується спільний простір

Матриця «запис × шматок» величезна й майже порожня. Ми стискаємо її
усіченим сингулярним розкладом (`TruncatedSVD`) до K чисел на запис. Кожен
рядок стає точкою у просторі з K вимірами.

Уся хитрість — **що покласти в рядок**:

- у режимі **B** рядок містить англійські **і** українські шматки одного
  запису. Слово `error` і слово `помилка` стоять в одному рядку **завжди**,
  тож розклад мусить дати їм схожі координати;
- у режимі **B0** ті самі тексти лежать у двох різних рядках. Розклад не має
  жодної підказки, що ці рядки пов'язані.

Далі документ будь-якої мови проєктується в цей простір **однією й тією ж**
матрицею. Перевіримо, що проєкція — це просто множення на матрицю, і жодної
магії там немає.

In [ ]:
from scipy import sparse
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.preprocessing import normalize


def to_counts(tok, texts, width):
    """Розріджена матриця «рядок × шматок» із частотами."""
    indptr, indices, data = [0], [], []
    for e in tok.encode_batch(texts):
        counted = Counter(e.ids)
        indices.extend(counted.keys())
        data.extend(counted.values())
        indptr.append(len(indices))
    return sparse.csr_matrix(
        (np.array(data, dtype=np.float32), np.array(indices), np.array(indptr)),
        shape=(len(texts), width))


# перевірка «наша реалізація = бібліотечна»: проєкція — це множення на матрицю
probe = to_counts(tok_shared, test_en[:200], tok_shared.get_vocab_size())
probe_svd = TruncatedSVD(n_components=16, random_state=0).fit(probe)
ours = probe.dot(probe_svd.components_.T)
assert np.allclose(ours, probe_svd.transform(probe), atol=1e-5), 'розійшлося!'
print('✅ проєкція в спільний простір — це множення на матрицю з',
      probe_svd.components_.shape[0], 'рядків, і нічого більше')
del probe, probe_svd, ours
_ = gc.collect()

### 10.2 · Гіперпараметри дібрані окремим прогоном

Розмір простору K і силу регуляризації C ми дібрали **на відкладеній
частині**, тим самим бюджетом у 30 000 пар, **однаковою сіткою для всіх трьох
режимів і обох мов** — інакше порівняння вимірювало б старанність, а не
методи. Сюди виписані готові числа; сам перебір — у домашньому завданні.

⚠️ Одна чесна деталь: на відкладеній K = 256 трохи кращий за K = 128 в обох
режимах, які його мають. Ми взяли 128, бо це бюджет зошита, і нижче
показуємо, що від K висновок не залежить.

In [ ]:
K_DIM = 128
BEST_C = {('A', 'en'): 64.0, ('A', 'uk'): 16.0,
          ('B', 'en'): 64.0, ('B', 'uk'): 256.0,
          ('B0', 'en'): 256.0, ('B0', 'uk'): 256.0}
SEEDS = [0, 1, 2, 3, 4]

print('розмір спільного простору K =', K_DIM)
print('зерен                       ', len(SEEDS))
print('дібрана сила регуляризації C по режимах і мовах:')
for key in sorted(BEST_C):
    print('   %-3s навчаємо на «%s»  C = %g' % (key[0], key[1], BEST_C[key]))


def prepare(seed):
    """Повертає ознаки трьох режимів для одного зерна."""
    test_i, dev_i, train_i = split_indices(seed)
    en = {k: [rows[i][1] for i in idx] for k, idx in
          [('tr', train_i), ('te', test_i)]}
    uk = {k: [rows[i][2] for i in idx] for k, idx in
          [('tr', train_i), ('te', test_i)]}
    tok = train_bpe(en['tr'] + uk['tr'], V_SHARED)
    width = tok.get_vocab_size()
    C_en = {k: to_counts(tok, v, width) for k, v in en.items()}
    C_uk = {k: to_counts(tok, v, width) for k, v in uk.items()}
    # idf спільний для обох мов — інакше мови жили б у різних шкалах
    tfidf = TfidfTransformer().fit(sparse.vstack([C_en['tr'], C_uk['tr']]))
    X_en = {k: tfidf.transform(v) for k, v in C_en.items()}
    X_uk = {k: tfidf.transform(v) for k, v in C_uk.items()}

    feats = {'A': (X_en, X_uk)}
    # B: обидві мови одного запису — в одному рядку
    svd_b = TruncatedSVD(n_components=K_DIM, random_state=seed)
    svd_b.fit(X_en['tr'] + X_uk['tr'])
    feats['B'] = ({k: normalize(svd_b.transform(v)) for k, v in X_en.items()},
                  {k: normalize(svd_b.transform(v)) for k, v in X_uk.items()})
    # B0: ті самі тексти, але мови в різних рядках
    svd_0 = TruncatedSVD(n_components=K_DIM, random_state=seed)
    svd_0.fit(sparse.vstack([X_en['tr'], X_uk['tr']]))
    feats['B0'] = ({k: normalize(svd_0.transform(v)) for k, v in X_en.items()},
                   {k: normalize(svd_0.transform(v)) for k, v in X_uk.items()})
    return feats, labels[train_i], labels[test_i], svd_b, svd_0


def one_run(features, y_train, y_test, mode, source):
    """Навчаємо на одній мові, перевіряємо на обох. Повертає (своя, чужа)."""
    X_en, X_uk = features[mode]
    train_X, other_X = (X_en, X_uk) if source == 'en' else (X_uk, X_en)
    clf = LogisticRegression(C=BEST_C[(mode, source)], max_iter=3000,
                             solver='liblinear')
    clf.fit(train_X['tr'], y_train)
    return (f1_score(y_test, clf.predict(train_X['te']), zero_division=0),
            f1_score(y_test, clf.predict(other_X['te']), zero_division=0))

Тепер сам прогін: пʼять зерен, кожне зерно — свій випадковий поділ, свій
словник, свій розклад. Пʼять, а не три, бо на цьому числі тримається головне
твердження теми, а три зерна дають **нижню оцінку** розкиду, а не розкид.

In [ ]:
started = time.process_time()
results = {}                    # (режим, мова-джерело, куди) -> список по зернах
alignment = {}                  # (режим, зерно) -> (косинус пар, косинус чужих)

for seed in SEEDS:
    features, y_train, y_test, svd_b, svd_0 = prepare(seed)
    for mode in ['A', 'B', 'B0']:
        for source in ['en', 'uk']:
            same, cross = one_run(features, y_train, y_test, mode, source)
            results.setdefault((mode, source, 'своя'), []).append(same)
            results.setdefault((mode, source, 'чужа'), []).append(cross)
    # заразом міряємо, чи стали переклади сусідами у просторі
    for mode in ['B', 'B0']:
        Zen, Zuk = features[mode][0]['te'], features[mode][1]['te']
        paired = float(np.mean(np.sum(Zen * Zuk, axis=1)))
        shifted = float(np.mean(np.sum(Zen * np.roll(Zuk, 1, axis=0), axis=1)))
        alignment[(mode, seed)] = (paired, shifted)
    del features, svd_b, svd_0
    gc.collect()
    print('зерно %d готове, минуло %.0f с процесорного часу'
          % (seed, time.process_time() - started))

In [ ]:
baseline = f1_score(labels[test_i], np.ones(len(test_i), dtype=int))
print('база «оголосити все помилкою»: F1 = %.4f' % baseline)
print()
NAMES = {'A': 'A · лише спільний словник',
         'B': 'B · простір, навчений на парах',
         'B0': 'B0 · той самий простір, мови нарізно'}
for source, target in [('en', 'укр'), ('uk', 'англ')]:
    home = 'англ' if source == 'en' else 'укр'
    print('НАВЧАЄМО НА МОВІ «%s», ПИТАЄМО ПРО «%s»' % (home, target))
    print('%-38s %18s %18s' % ('режим', 'своя мова', 'чужа мова'))
    for mode in ['A', 'B', 'B0']:
        s = np.array(results[(mode, source, 'своя')])
        c = np.array(results[(mode, source, 'чужа')])
        print('%-38s %8.4f ±%.4f %8.4f ±%.4f'
              % (NAMES[mode], s.mean(), s.std(), c.mean(), c.std()))
    print()

### 10.3 · Купи, а не середні

Курс домовився давно: різниця, менша за розкид, не є різницею. Тож друкуємо
не середні, а **купи** — від найгіршого зерна до найкращого.

In [ ]:
print('ПЕРЕНОС «англійська → українська», пʼять зерен')
print('%-38s %10s %10s %s' % ('режим', 'найгірше', 'найкраще', 'середнє'))
piles = {}
for mode in ['A', 'B', 'B0']:
    v = np.array(results[(mode, 'en', 'чужа')])
    piles[mode] = (v.min(), v.max())
    print('%-38s %10.4f %10.4f %8.4f' % (NAMES[mode], v.min(), v.max(), v.mean()))

print()
for a, b in [('B', 'A'), ('B', 'B0')]:
    overlap = not (piles[a][1] < piles[b][0] or piles[b][1] < piles[a][0])
    print('купи %s і %s %s' % (a, b, 'ПЕРЕКРИВАЮТЬСЯ' if overlap
                               else 'не перетинаються'))

print()
print('ПЕРЕНОС «українська → англійська», пʼять зерен')
print('%-38s %10s %10s %s' % ('режим', 'найгірше', 'найкраще', 'середнє'))
for mode in ['A', 'B', 'B0']:
    v = np.array(results[(mode, 'uk', 'чужа')])
    print('%-38s %10.4f %10.4f %8.4f' % (NAMES[mode], v.min(), v.max(), v.mean()))

### 10.4 · Чи справді переклади стали сусідами

Перенос мав би працювати тому, що переклад лягає у просторі поруч зі своїм
оригіналом. Перевіримо це прямо: порахуємо косинус між проєкцією
англійського запису й проєкцією **його власного** перекладу, а для порівняння
— косинус із **чужим** перекладом (зсунутим на один рядок).

In [ ]:
print('%-38s %14s %14s %10s' %
      ('режим', 'свій переклад', 'чужий переклад', 'розрив'))
for mode in ['B', 'B0']:
    p = np.array([alignment[(mode, s)][0] for s in SEEDS])
    r = np.array([alignment[(mode, s)][1] for s in SEEDS])
    print('%-38s %8.4f ±%.4f %8.4f ±%.4f %10.4f'
          % (NAMES[mode], p.mean(), p.std(), r.mean(), r.std(),
             p.mean() - r.mean()))

### 10.5 · Чи залежить висновок від розміру простору

K — це місткість спільного простору, тобто рівно та величина, яку в справжніх
багатомовних моделях ділять між мовами. Прогонимо режим B на чотирьох
значеннях K, на одному зерні, і подивимось, чи не тримається наш висновок на
випадковому виборі.

In [ ]:
started = time.process_time()
test_i, dev_i, train_i = split_indices(0)
en = {'tr': [rows[i][1] for i in train_i], 'te': [rows[i][1] for i in test_i]}
uk = {'tr': [rows[i][2] for i in train_i], 'te': [rows[i][2] for i in test_i]}
tok_k = train_bpe(en['tr'] + uk['tr'], V_SHARED)
width = tok_k.get_vocab_size()
Cen = {k: to_counts(tok_k, v, width) for k, v in en.items()}
Cuk = {k: to_counts(tok_k, v, width) for k, v in uk.items()}
tfidf_k = TfidfTransformer().fit(sparse.vstack([Cen['tr'], Cuk['tr']]))
Xen = {k: tfidf_k.transform(v) for k, v in Cen.items()}
Xuk = {k: tfidf_k.transform(v) for k, v in Cuk.items()}
y_tr, y_te = labels[train_i], labels[test_i]

print('%6s %14s %14s' % ('K', 'англ → англ', 'англ → укр'))
for K in [32, 64, 128, 256]:
    svd = TruncatedSVD(n_components=K, random_state=0).fit(Xen['tr'] + Xuk['tr'])
    Zen = {k: normalize(svd.transform(v)) for k, v in Xen.items()}
    Zuk = {k: normalize(svd.transform(v)) for k, v in Xuk.items()}
    clf = LogisticRegression(C=BEST_C[('B', 'en')], max_iter=3000,
                             solver='liblinear').fit(Zen['tr'], y_tr)
    print('%6d %14.4f %14.4f'
          % (K, f1_score(y_te, clf.predict(Zen['te'])),
             f1_score(y_te, clf.predict(Zuk['te']))))
    del svd, Zen, Zuk, clf
    gc.collect()
print('зайняло %.0f с процесорного часу' % (time.process_time() - started))

## 11 · Що з цього виходить

Збери докупи:

1. **Націнка в токенах реальна й вимірна.** На тому самому змісті українська
   коштує більше токенів за англійську, і націнка падає, коли словнику дають
   місце.
2. **Спільний словник коштує обом мовам.** Той самий бюджет, поділений
   надвоє, дає кожній гірше різання — це і є прокляття багатомовності в
   найдешевшій формі.
3. **Спільний словник сам собою переносу не дає.** Режим A на чужій мові
   провалюється нижче за базу «оголосити все помилкою».
4. **Перенос дає спільний сигнал.** Той самий розклад, навчений на парах,
   переносить; навчений на тих самих текстах нарізно — ні.

⚠️ І межа, про яку треба сказати вголос. Наш «спільний простір» — це лінійний
розклад, а не трансформер; наші дві мови — не сто; наші переклади ідеально
паралельні, а справжні багатомовні моделі вчаться переважно **без** паралельних
пар. Тому число «перенос дає стільки-то» — це число **нашого** заміру, а не
властивість mBERT. Що переноситься дослівно — це **механізм**: спільність
шматків не звʼязує мов, звʼязує спільний контекст.

In [ ]:
print('зошит зайняв %.0f с процесорного часу' % (time.process_time() - ЗАПУСК))
print('навантаження машини наприкінці', round(os.getloadavg()[0], 2))

## Завдання

🟢 **Рівень 1.** Додай до таблиці розділу 4 ще один розмір словника — 64 000 —
і скажи, чи націнка на українську впала нижче за 25 %.

🟡 **Рівень 2.** Повтори розділ 7 для спільного словника на 32 000 шматків.
Частка українських вживань спільними шматками зросла чи впала? Поясни знак.

🔴 **Рівень 3.** Ми взяли готові C і K із окремого добору. Зроби добір сам:
сітка C з пʼяти значень і K з трьох, **на відкладеній частині** (`dev_i`),
однакова для всіх трьох режимів. Чи змінився порядок режимів за переносом?